In [2]:
# ------------------------------------
#  Mount Google Drive
# ------------------------------------

# Import library to access Google Drive in Colab
from google.colab import drive

# Mount Google Drive to access files stored in your drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import torch
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from torch.utils.data import DataLoader
import pandas as pd


In [4]:
# Load test data
test_df = pd.read_csv('/content/drive/MyDrive/Fake News Detection /Processed Data/test_dataset.csv')
X_test = test_df['content'].fillna("").astype(str).tolist()
y_test = torch.tensor(test_df['label'].values, dtype=torch.long)

In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_path = ("/content/drive/MyDrive/Fake News Detection /Processed Data/Model Training")

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

model.eval()

print("Loaded successfully")

Loading weights:   0%|          | 0/104 [00:01<?, ?it/s]

Loaded successfully


In [6]:
from transformers import AutoTokenizer

model_path = ("/content/drive/MyDrive/Fake News Detection /Processed Data/Model Training")

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)

In [7]:
# -------------------------------
# 4. Tokenization for Testing
# -------------------------------
# Tokenize test data
test_encodings = tokenizer(
    X_test,
    truncation=True,
    padding=True,
    max_length=256,
    return_tensors="pt"
)
input_ids = test_encodings['input_ids']
attention_mask = test_encodings['attention_mask']



In [8]:
# 5. Batch Prediction

batch_size = 16
predictions = []

for i in range(0, len(input_ids), batch_size):

    batch_input_ids = input_ids[i:i+batch_size].to(device)
    batch_attention_mask = attention_mask[i:i+batch_size].to(device)

    with torch.no_grad():
        outputs = model(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_mask
        )

    logits = outputs.logits
    preds = torch.argmax(logits, dim=1).cpu().numpy()

    predictions.extend(preds)

In [15]:
# 6. Evaluation
from sklearn.metrics import confusion_matrix
accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)
f1 = f1_score(y_test, predictions)

print("\n===== Evaluation Results =====")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)


===== Evaluation Results =====
Accuracy : 0.9915367483296214
Precision: 0.9856879039704525
Recall   : 0.9967320261437909
F1 Score : 0.9911792014856081


In [16]:
# -------------------------------
# 7. Confusion Matrix
# -------------------------------
cm = confusion_matrix(y_test, predictions)
print("\nConfusion Matrix:\n", cm)


Confusion Matrix:
 [[4634   62]
 [  14 4270]]


In [17]:
# -------------------------------
# 8. Detailed Report
# -------------------------------
print("\nClassification Report:\n")
print(classification_report(y_test, predictions))


Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.99      0.99      4696
           1       0.99      1.00      0.99      4284

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980



In [ ]:
# 9. PREDICTION FUNCTION (USER INPUT)


def predict_news(text):

    inputs = tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    pred = torch.argmax(logits, dim=1).item()

    if pred == 1:
        return "REAL NEWS ✅"
    else:
        return "FAKE NEWS ❌"


# -------------------------------
# 10. Take User Input
# -------------------------------
while True:
    user_input = input("\nEnter news (type 'exit' to stop): ")

    if user_input.lower() == "exit":
        break

    result = predict_news(user_input)
    print("Prediction:", result)


Enter news (type 'exit' to stop): "SHOCKING: Top scientists confirm that 5G towers are actually transmitting the virus! The radiation weakens your immune system while spreading the infection directly to your lungs. Turn off your phones immediately! DO NOT WATCH THE NEWS, THEY ARE LYING! SHARE BEFORE THIS IS TAKEN DOWN!"
Prediction: FAKE NEWS ❌

Enter news (type 'exit' to stop): "SHOCKING: Top scientists confirm that 5G towers are actually transmitting the virus! The radiation weakens your immune system while spreading the infection directly to your lungs. Turn off your phones immediately! DO NOT WATCH THE NEWS, THEY ARE LYING! SHARE BEFORE THIS IS TAKEN DOWN!"
Prediction: FAKE NEWS ❌

Enter news (type 'exit' to stop): "BREAKING: Senator Smith Caught Selling Secret Documents at Midnight Meeting! (Video proof inside). The mainstream media won't tell you this, but we have the exclusive story that will change the election tomorrow." 
Prediction: FAKE NEWS ❌

Enter news (type 'exit' to sto